In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

# import sys
# sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import functions.mdata_utils as mdata_utils
import functions.TCR_embedings as TCR_embedings

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA

In [3]:
importlib.reload(TCR_embedings)

<module 'functions.TCR_embedings' from 'e:\\Python code\\Machine learning\\JupyterNote\\Bio_CRC\\Data processing\\functions\\TCR_embedings.py'>

In [ ]:
import re
data_path = "/ix1/ylee/Yifan_Zhang/Code_data/external"
folders = [f for f in current_dir.iterdir() if f.is_dir() and not f.name.startswith('_')]

# Load yz_processed.h5mu from each folder into named variables
mdata_dict = {}
mdata_list = []

type_sets = {
    "CD4"     : ['Cd4'],
    "CD8"     : ['Cd8a', 'Cd8b1','Nkg7'],
}

CD4_cell_types = {
    "Treg"    : ['Foxp3', 'Il2ra', 'Ctla4', 'Ikzf2'],
    "Th1"     : ['Tbx21', 'Stat1', 'Stat4', 'Ifng', 'Tnf', 'Il2', 'Cxcr3', 'Ccr5', 'Il12rb1', 'Il12rb2', 'Il18r1'],
    "Th2"     : ['Gata3', 'Stat6', 'Il4', 'Il5', 'Il13', 'Ccr4', 'Ccr8', 'Il1rl1', 'Ptgdr2', 'Areg'],
    "Th17"    : ['Rorc', 'Rora', 'Stat3', 'Batf', 'Il17a', 'Il17f', 'Il22', 'Il21', 'Ccr6', 'Il23r'],
}

common_states = {
    "Naive"         : ['Ccr7', 'Sell', 'Tcf7', 'Lef1', 'Il7r'],   # both CD4 and CD8 naive
    "Cytotoxic"     : ['Gzmb', 'Gzma', 'Gzmk', 'Prf1', 'Nkg7'],  # Tem_CD8, SLEC, Tex_term, Th1-cytotoxic
    "Exhaustion_core": ['Pdcd1', 'Tox', 'Lag3', 'Tigit', 'Havcr2'],# shared across Tpex, Tex_int, Tex_term
    "Tcf7_stem"     : ['Tcf7', 'Bcl2', 'Id3', 'Bach2'],           # Tpex, MPEC, Naive
    "Proliferating" : ['Mki67', 'Top2a', 'Tyms', 'Cdk1'],         # any cycling T cell
    "IFN_stim"      : ['Isg15', 'Ifit1', 'Ifit3', 'Mx1', 'Oas1a', 'Gbp2'],
    "Early_activ"   : ['Cd69','Cd28', 'Icos', 'Nr4a1', 'Nr4a2', 'Fos', 'Jun'],
    "Memory_core"   : ['Il7r', 'Bcl2', 'S100a4', 'Ccl5'],         # Tcm, Tem, MPEC
    "Effector_core" : ['Cx3cr1', 'S1pr1', 'Zeb2', 'Tbx21'],       # SLEC, Tem_CD8, Tex_KLR
}

all_important_genes = []
for gene_list in type_sets.values():
    all_important_genes.extend(gene_list)
for gene_list in state_sets.values():
    all_important_genes.extend(gene_list)
    
gse_to_cell_type = {
    'GSE156718': 'CD4',
    # 'GSE182747': ,  # Adjust based on actual cell type
    'GSE293883': 'CD4',
    'GSE178085': 'Th17',
    'GSE188320': 'Th17',
    # 'GSE1': 'CD4',  # Adjust based on actual cell type (LEE)
}

In [ ]:
n_top_genes = 2000
for folder in folders[0]:
    h5mu_file = folder / 'yz_processed_allGenes_annotateByScore.h5mu'
    
    if h5mu_file.exists():
        print(f"Loading: {h5mu_file}")
        mdata = mu.read(h5mu_file)
        var_name = re.sub(r'[^\w]', '_', folder.name) + '_mdata'
        mdata = mdata_utils.pp_EAE(mdata)
        
        sc.pp.highly_variable_genes(mdata['gex'], n_top_genes=n_top_genes)
        
        # Check if genes in type_sets or state_sets were removed and add them back
        # Get genes that exist in the data but are not marked as highly variable
        existing_important_genes = [g for g in all_important_genes if g in mdata['gex'].var_names]
        removed_genes = [g for g in existing_important_genes if not mdata['gex'].var.loc[g, 'highly_variable']]
        
        if removed_genes:
            print(f"  Adding back {len(removed_genes)} removed important genes: {removed_genes}")
            mdata['gex'].var.loc[removed_genes, 'highly_variable'] = True
       
        # reduce number of vars
        mdata.mod['gex'] = mdata['gex'][:, mdata['gex'].var['highly_variable']].copy()
        mdata = mdata_utils.sync_mdata_obs(mdata)

        # Annotate if cloned cells appear in many locations
        clone_thresh = 2
        mdata.obs['cloned'] = mdata['airr'].obs['clone_id_size'] >= clone_thresh

        # Check if appear in two tissues
        mdata_cloned = mdata[mdata.obs['cloned']]
        clone_tissue_df = mdata_cloned['airr'].obs[['clone_id']].join(mdata_cloned['gex'].obs['tissue'])
        clone_tissue_counts = clone_tissue_df.groupby('clone_id')['tissue'].nunique()
        multi_tissue_clones = clone_tissue_counts[clone_tissue_counts > 1].index
        mdata.obs['in_two_tissue'] = mdata['airr'].obs['clone_id'].isin(multi_tissue_clones)
        
    else:
        print(f"Skipping {folder.name}: yz_processed.h5mu not found")

In [ ]:
mdata

MuData object with n_obs × n_vars = 125524 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	125524 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition'
      obsm:	'airr', 'chain_indices'
    gex:	125524 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

In [7]:
mdata['airr'].obs['GSE'] = mdata.obs['GSE']
mdata['airr'].obs['GSE'].value_counts()

GSE
LEE          43558
GSE182747    37164
GSE188320    22346
GSE293883    11097
GSE156718    10848
GSE178085      511
Name: count, dtype: int64

# Cluster

In [ ]:
sc.pp.pca(mdata["gex"], n_comps=50)
sc.pp.neighbors(mdata["gex"], n_neighbors = 50)
sc.tl.umap(mdata["gex"], min_dist=0.5, spread=1.0)


MuData object with n_obs × n_vars = 105919 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length'
  obsm:	'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_atchley_pairwise', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_atchley_pairwise', 'X_VJ_1_cdr3_aa_composition', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call', 'tcr_embs'
  2 modalities
    airr:	105919 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	105919 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap', 'Tissue_group_colors'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

In [ ]:
sc.pl.umap(mdata["gex"], color=['state', 'cell_type'])

# embed TCR AA into vector

In [34]:
min_clone = 1
cloned_mask = mdata['airr'].obs['clone_id_size'].fillna(0).astype('int') > min_clone
mdata_cloned = mdata[cloned_mask].copy()
mdata_single = mdata[~cloned_mask].copy()
mdata_cloned['airr'].obs['GSE'].value_counts()

GSE
GSE182747    21993
GSE188320    12436
LEE           6001
GSE156718     2892
GSE293883     2293
GSE178085      501
Name: count, dtype: int64

In [37]:
# Embed TCR AA sequences using Atchley factors and one-hot encoding
tcr_aa_obs = ['VDJ_1_cdr3_aa', 'VJ_1_cdr3_aa']
tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call']

mdata = TCR_embedings.embed_tcr_aa(
    mdata,
    tcr_aa_obs=tcr_aa_obs,
    tcr_cat_features=tcr_cat_features,
    min_length=7, 
    max_length=22
)

Stored VDJ_1_cdr3_aa lengths in mdata.obs['VDJ_1_cdr3_aa_length']
Stored VDJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VDJ_1_cdr3_aa_atchley'] with shape (105919, 100)
Stored VDJ_1_cdr3_aa adjacent Atchley interactions in mdata.obsm['X_VDJ_1_cdr3_aa_atchley_pairwise'] with shape (105919, 475)
Stored VDJ_1_cdr3_aa AA composition in mdata.obsm['X_VDJ_1_cdr3_aa_composition'] with shape (105919, 20)

Stored VJ_1_cdr3_aa lengths in mdata.obs['VJ_1_cdr3_aa_length']
Stored VJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VJ_1_cdr3_aa_atchley'] with shape (105919, 100)
Stored VJ_1_cdr3_aa adjacent Atchley interactions in mdata.obsm['X_VJ_1_cdr3_aa_atchley_pairwise'] with shape (105919, 475)
Stored VJ_1_cdr3_aa AA composition in mdata.obsm['X_VJ_1_cdr3_aa_composition'] with shape (105919, 20)

VDJ_1_j_call: 12 unique categories + 1 unknown = 13 dimensions
VDJ_1_v_call: 23 unique categories + 1 unknown = 24 dimensions
VJ_1_j_call: 42 unique categories + 1 unknown = 43 dimensions
VJ_1_v_call: 105 u

In [43]:
mdata.obsm['VDJ_1_v_call']

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(105919, 24))

In [44]:
arrs_tcr = []
for key, value in mdata.obsm.items():
    arrs_tcr.append(value)

In [45]:
view_tcr = np.concatenate(arrs_tcr, axis=1)
print(view_tcr.shape)

# Add chain length
for chain in tcr_aa_obs:
    view_tcr = np.concatenate([view_tcr, mdata.obs[chain + '_length'].to_numpy().reshape(-1, 1)],  axis=1)

# Add clone size
view_tcr = np.concatenate([view_tcr, mdata['airr'].obs['clone_id_size'].to_numpy().reshape(-1, 1)],  axis=1)
    
print(view_tcr.shape)

(105919, 2759)
(105919, 2762)


In [46]:
# Perform Canonical Correlation Analysis separately on train and test sets
view_gene = mdata['gex'].X.toarray()
# view_gene = mdata['gex'].obsm['X_pca_harmony']

scaler = StandardScaler()
view_tcr = scaler.fit_transform(view_tcr)
view_gene = scaler.fit_transform(view_gene)

In [47]:
mdata.obsm['tcr_embs'] = view_tcr

## Save

In [48]:
importlib.reload(TCR_embedings)

<module 'functions.TCR_embedings' from 'e:\\Python code\\Machine learning\\JupyterNote\\Bio_CRC\\Data processing\\functions\\TCR_embedings.py'>

In [ ]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True
# mdata.write(f'{filename}_singlets_CVscores.h5mu')